In [2]:
!pip install -q transformers datasets sentencepiece
!pip install pandas nltk bnlp_toolkit


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 74.1 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of bnlp-toolkit to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.4/175.4 kB 7.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 41.5 MB/s eta 0:00:00
  Created wheel for emoji: filename=emoji-1.7.0-py3-none-any.whl size=171031 sha256=124c00b719d94868952b158031125795c9f79dac0c618c0ff53145283721e407
  Stored in directory: /root/.cache/pip/wheels/e0/8c/e0/294d2e4ea0e55792bfc99b6b263e4a0511443da7b69af67688
Successfully built emoji
  Attempting uninstall: emoji
    Found existing installation: emoj

In [3]:
import pandas as pd
import glob

path = "/kaggle/input/vashantor/Vashantor/Train/"
all_files = glob.glob(path + "*.csv")

df_list = []

for file in all_files:
    df = pd.read_csv(file)
    
    # Clean column names: remove leading/trailing spaces and make lowercase
    df.columns = df.columns.str.strip().str.lower()
    
    # 1. Identify the dialect column (the one with the region name in it)
    # We look for something that ends with '_bangla_speech' but isn't just 'bangla_speech'
    regional_col = [c for c in df.columns if '_bangla_speech' in c and c != 'bangla_speech']
    
    if regional_col:
        df['dialect_speech'] = df[regional_col[0]]
    else:
        # Fallback: If no specific dialect column found, create it as empty to avoid KeyError
        df['dialect_speech'] = None

    # 2. Map the columns we need, using a safety check to avoid KeyError
    # We create a new dataframe with exactly the columns you want
    processed_df = pd.DataFrame()
    processed_df['region_name'] = df['region_name'] if 'region_name' in df.columns else [None]*len(df)
    processed_df['bangla_speech'] = df['bangla_speech'] if 'bangla_speech' in df.columns else [None]*len(df)
    processed_df['english_speech'] = df['english_speech'] if 'english_speech' in df.columns else [None]*len(df)
    processed_df['dialect_speech'] = df['dialect_speech']

    df_list.append(processed_df)

# Combine all dataframes
combined_df = pd.concat(df_list, ignore_index=True)

# Shuffle
combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)



In [4]:
df_train = combined_df

In [5]:
path = "/kaggle/input/vashantor/Vashantor/Test/"
all_files = glob.glob(path + "*.csv")

df_list = []

for file in all_files:
    df = pd.read_csv(file)
    
    # Clean column names: remove leading/trailing spaces and make lowercase
    df.columns = df.columns.str.strip().str.lower()
    
    # 1. Identify the dialect column (the one with the region name in it)
    # We look for something that ends with '_bangla_speech' but isn't just 'bangla_speech'
    regional_col = [c for c in df.columns if '_bangla_speech' in c and c != 'bangla_speech']
    
    if regional_col:
        df['dialect_speech'] = df[regional_col[0]]
    else:
        # Fallback: If no specific dialect column found, create it as empty to avoid KeyError
        df['dialect_speech'] = None

    # 2. Map the columns we need, using a safety check to avoid KeyError
    # We create a new dataframe with exactly the columns you want
    processed_df = pd.DataFrame()
    processed_df['region_name'] = df['region_name'] if 'region_name' in df.columns else [None]*len(df)
    processed_df['bangla_speech'] = df['bangla_speech'] if 'bangla_speech' in df.columns else [None]*len(df)
    processed_df['english_speech'] = df['english_speech'] if 'english_speech' in df.columns else [None]*len(df)
    processed_df['dialect_speech'] = df['dialect_speech']

    df_list.append(processed_df)

# Combine all dataframes
combined_df = pd.concat(df_list, ignore_index=True)

# Shuffle
combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)



In [6]:
test_df= combined_df

In [7]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from tqdm import tqdm

In [8]:
# Load data

df= df_train

df['region_name'] = df['region_name'].str.lower().str.strip()



# # Extract dialect from region_name and create label mapping
# df['dialect'] = df['region_name'].str.lower()
# dialect_to_id = {dialect: idx for idx, dialect in enumerate(df['dialect'].unique())}
# id_to_dialect = {idx: dialect for dialect, idx in dialect_to_id.items()}
# df['label'] = df['dialect'].map(dialect_to_id)

In [9]:
len(df_train)

9375

In [10]:
sources = []
targets = []

for _, row in df.iterrows():
    dialect = row['region_name']
    src_col = f"dialect_speech"
    tgt_col = f"bangla_speech"

    if src_col in df.columns and tgt_col in df.columns:
        if pd.notna(row[src_col]) and pd.notna(row[tgt_col]):
            sources.append(str(row[src_col]))
            targets.append(str(row[tgt_col]))

print("Total samples:", len(sources))


Total samples: 9375


In [11]:
len(test_df)

1875

In [2]:
# import torch
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# # 1. Set the new model name
# MODEL_NAME = "facebook/nllb-200-distilled-600M"

# # 2. Load the tokenizer and model
# # For NLLB, it's often helpful to specify the source language (src_lang) at initialization
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang="ben_Beng")
# model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)

# # --- Example Usage (Translating Bengali to English) ---

# text = "আমি কৃত্রিম বুদ্ধিমত্তা নিয়ে কাজ করছি।"  # "I am working with AI."

# # Prepare inputs
# inputs = tokenizer(text, return_tensors="pt").to(device)

# # Generate translation 
# # We specify 'eng_Latn' (English) as the forced target language
# # 1. Identify the target language code
# target_lang_code = "eng_Latn"

# # 2. Get the token ID safely
# forced_bos_token_id = tokenizer.convert_tokens_to_ids(target_lang_code)

# # 3. Generate
# translated_tokens = model.generate(
#     **inputs, 
#     forced_bos_token_id=forced_bos_token_id, 
#     max_length=128
# )

# # Decode results
# output = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]
# print(f"Translation: {output}")

Translation: I'm working on artificial intelligence.


In [12]:
# MODEL_NAME = "csebuetnlp/banglat5_small"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)


tokenizer_config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/646 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
2026-01-08 12:47:24.498272: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767876444.785043      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767876444.871813      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has

pytorch_model.bin:   0%|          | 0.00/242M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [13]:
class DialectNormalizeDataset(Dataset):
    def __init__(self, sources, targets, tokenizer, max_len=128):
        self.sources = sources
        self.targets = targets
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.sources)

    def __getitem__(self, idx):
        src = self.sources[idx]
        tgt = self.targets[idx]

        source_enc = self.tokenizer(
            src,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        target_enc = self.tokenizer(
            tgt,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        labels = target_enc["input_ids"]
        labels[labels == tokenizer.pad_token_id] = -100

        return {
            "input_ids": source_enc["input_ids"].squeeze(),
            "attention_mask": source_enc["attention_mask"].squeeze(),
            "labels": labels.squeeze()
        }


In [14]:
dataset = DialectNormalizeDataset(sources, targets, tokenizer)
loader = DataLoader(dataset, batch_size=8, shuffle=True)


In [15]:
optimizer = AdamW(model.parameters(), lr=5e-5)
epochs = 5

total_steps = len(loader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)


In [ ]:
model.train()

for epoch in range(epochs):
    total_loss = 0

    for batch in tqdm(loader, desc=f"Epoch {epoch+1}"):
        optimizer.zero_grad()

        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss/len(loader):.4f}")



Epoch 1: 100%|██████████| 1172/1172 [1:42:17<00:00,  5.24s/it]


Epoch 1 Loss: 4.5694


Epoch 2:  97%|█████████▋| 1139/1172 [1:39:35<02:54,  5.28s/it]

In [ ]:
def normalize_dialect(text):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=64,
            num_beams=5
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [ ]:
print(normalize_dialect("তোইয়ার হতা বলার ধরণ বহুত সুন্দর"))


In [ ]:
test_df['normalized_output'] = test_df['dialect_speech'].apply(normalize_dialect)

test_df.to_csv('normalized_results.csv', index=False, encoding='utf-8')

# print(test_df[['dialect_speech', 'normalized_output']].head())

In [ ]:
from nltk.translate.bleu_score import corpus_bleu
 

# Tokenize the sentences (BLEU works on lists of tokens)
references = [[ref.split()] for ref in test_df['bangla_speech']]
candidates = [can.split() for can in test_df['normalized_output']]

# Calculate Corpus BLEU
score = corpus_bleu(references, candidates)
# 
print(f"Cumulative BLEU Score: {score * 100:.2f}")

In [ ]:
import pandas as pd
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

# 1. Initialize Smoothing
chencherry = SmoothingFunction()

# 2. Prepare the data (Ensure it's a list of words)
# We use .split() to make sure we aren't comparing character-by-character
candidates = [str(text).split() for text in test_df['normalized_output']]
references = [[str(text).split()] for text in test_df['bangla_speech']]

# 3. Calculate with smoothing
# weights=(0.5, 0.5, 0, 0) focuses on 1-gram and 2-gram matches, better for short sentences
score = corpus_bleu(references, candidates, smoothing_function=chencherry.method1)

print(f"Corrected BLEU Score: {score * 100:.2f}")

In [ ]:
print(f"Original: {test_df['bangla_speech'].iloc[0]}")
print(f"Output:   {test_df['normalized_output'].iloc[0]}")
print(f"Target:   {test_df['bangla_speech'].iloc[0]}")

In [ ]:
model.save_pretrained("/kaggle/working/dialect_normalizer")
tokenizer.save_pretrained("/kaggle/working/dialect_normalizer")
